---
last_verified: 2026-09-30
tool_version: n/a
---

# Choosing between the Falco rule engine and eBPF probes

> Runtime-detection reference. Release-neutral by design: no Falco version is
> claimed here, and nothing below depends on behaviour specific to one release.

## Purpose

Falco's runtime pipeline has two decision points that are easy to conflate:
**where an event is collected**, and **where the decision to alert is made**.

The **rule engine** evaluates conditions in userspace. Collected events arrive
with all the context the kernel was able to attach, every enabled rule is
evaluated against each event, and an alert is emitted for each rule whose
conditions hold. Nothing is discarded before that point.

An **eBPF probe** moves part of the decision earlier. The probe program itself
carries a predicate; events that fail it are dropped in the kernel and never
cross into userspace. What reaches the rule engine is already narrowed, and
whatever the predicate did not encode is simply not available downstream.

This notebook compares the two shapes on coverage, context availability, change
management, and diagnostic difficulty, then supplies a scoring function for
choosing between them for a given workload.

## When to use

Reach for this comparison when deciding:

- whether broad, generic rules should evaluate every collected event, or whether
  a narrow probe should pre-filter so the engine sees only candidates;
- whether a detection depends on context (user identity, full command line,
  container metadata) that a kernel-side predicate has to approximate;
- how rule changes will be delivered — a userspace ruleset swap versus
  re-attaching a loaded program;
- how much of the event volume a workload is expected to generate, and what
  that volume costs when every event is carried across the kernel/userspace
  boundary.

## Prerequisites

- Python 3, standard library only.
- A Linux host. Falco's kernel-side collection is Linux-only, so the surrounding
  comparison is written against Linux concepts rather than a portable subset.
- Familiarity with the shape of a Falco detection: a rule name, a priority, a
  condition over event fields, and a human-readable output line.

## Steps

### 1. Model the userspace rule engine

The engine is the general-purpose component. It receives every event the driver
collects and applies each rule's condition in turn, emitting one alert per
matching rule. The cell below builds a small illustrative ruleset and runs a
representative event stream through it.

In [ ]:
# last_verified: 2026-09-30 · falco n/a

# Userspace rule engine model.
# Field names are illustrative, not a schema claim.

RULES = [
    {
        "name": "illustrative-shell-in-container",
        "priority": "Warning",
        "condition": lambda e: e["container"] is not None
        and e["proc"] in {"bash", "sh"},
    },
    {
        "name": "illustrative-credential-file-write",
        "priority": "Critical",
        "condition": lambda e: e["proc"] in {"cat", "tee"}
        and "/etc/shadow" in e["cmdline"],
    },
    {
        "name": "illustrative-package-install-in-container",
        "priority": "Notice",
        "condition": lambda e: e["container"] is not None
        and e["proc"] in {"apt", "apt-get", "yum", "dnf"},
    },
]


def evaluate(events, rules):
    """Return one alert per (event, matching rule) pair."""
    alerts = []
    for event in events:
        for rule in rules:
            if rule["condition"](event):
                alerts.append(
                    {
                        "rule": rule["name"],
                        "priority": rule["priority"],
                        "pid": event["pid"],
                        "container": event["container"],
                    }
                )
    return alerts


EVENT_STREAM = [
    {"pid": 1001, "proc": "nginx", "container": None, "cmdline": "nginx -g daemon off;"},
    {"pid": 1002, "proc": "bash", "container": "web-1", "cmdline": "/bin/bash"},
    {"pid": 1003, "proc": "curl", "container": "web-1", "cmdline": "curl -s /tmp/data.json"},
    {"pid": 1004, "proc": "apt", "container": "web-1", "cmdline": "apt-get install -y curl"},
    {"pid": 1005, "proc": "cat", "container": None, "cmdline": "cat /etc/shadow"},
]

alerts = evaluate(EVENT_STREAM, RULES)
print("events in:", len(EVENT_STREAM), "alerts out:", len(alerts))
for a in alerts:
    print(" ", a["priority"], a["rule"], "pid", a["pid"])

# Every collected event was available to every rule.
assert len(EVENT_STREAM) == 5
assert {a["pid"] for a in alerts} == {1002, 1004, 1005}


### 2. Model the eBPF probe as a kernel-side pre-filter

The probe changes what the engine can ever see. Its predicate runs against a
narrow set of kernel-attached fields, and non-matching events are dropped before
the boundary crossing. Two consequences follow, and both are visible in the
cell: the userspace event volume collapses, and any detection that needed a
field outside the predicate's reach can no longer fire.

In [ ]:
# eBPF probe model: a kernel-side predicate decides what crosses the
# boundary. A realistic probe predicate is expressible over a fixed, small set
# of fields — process name, container identity, limited argument slices.

PROBE_PREDICATE = lambda e: e["container"] is not None  # container-scoped only


def probe_forward(events, predicate):
    forwarded = [e for e in events if predicate(e)]
    dropped = len(events) - len(forwarded)
    return forwarded, dropped


forwarded, dropped = probe_forward(EVENT_STREAM, PROBE_PREDICATE)
print("forwarded to userspace:", len(forwarded), "dropped in kernel:", dropped)

# The engine now sees a strictly smaller stream ...
assert len(forwarded) + dropped == len(EVENT_STREAM)
assert all(e["container"] is not None for e in forwarded)

# ... and the host-level detection it used to make is now impossible.
kernel_filtered_alerts = evaluate(forwarded, RULES)
fired_rules = {a["rule"] for a in kernel_filtered_alerts}
print("rules that can still fire:", sorted(fired_rules))

# The host-level credential-read rule lost its only matching event.
assert "illustrative-credential-file-write" not in fired_rules
assert "illustrative-shell-in-container" in fired_rules

# Both host-level events (the nginx start and the credential read) were
# dropped in the kernel, so no rule ever saw them.
dropped_pids = {e["pid"] for e in EVENT_STREAM if e not in forwarded}
print("dropped pids:", sorted(dropped_pids))
assert dropped_pids == {1001, 1005}


### 3. What the two shapes cost as rule sets grow

Coverage behaves differently in each shape. In the engine model, a new rule is
free in terms of ingestion — it sees the whole stream immediately. In the probe
model, adding a detection means widening the kernel-side predicate, and widening
it increases the volume that crosses the boundary for *every* downstream rule.
The cell measures that trade-off directly.

In [ ]:
def probe_cost(events, predicate, rules):
    """Return (alerts found, events crossing the boundary)."""
    forwarded, _ = probe_forward(events, predicate)
    return evaluate(forwarded, rules), len(forwarded)


WIDE_PREDICATE = lambda e: e["proc"] in {"bash", "sh", "apt", "apt-get", "cat", "tee"}
NARROW_PREDICATE = lambda e: e["proc"] in {"bash", "sh"}

# Narrow predicate: cheaper boundary, fewer detections.
narrow_alerts, narrow_cost = probe_cost(EVENT_STREAM, NARROW_PREDICATE, RULES)
# Wide predicate: more detections, but every one of them costs boundary traffic.
wide_alerts, wide_cost = probe_cost(EVENT_STREAM, WIDE_PREDICATE, RULES)
engine_alerts, engine_cost = probe_cost(EVENT_STREAM, lambda e: True, RULES)

print(f"{'mode':<10}{'alerts':>8}{'events forwarded':>19}")
print(f"{'narrow':<10}{len(narrow_alerts):>8}{narrow_cost:>19}")
print(f"{'wide':<10}{len(wide_alerts):>8}{wide_cost:>19}")
print(f"{'engine':<10}{len(engine_alerts):>8}{engine_cost:>19}")

assert narrow_cost <= wide_cost <= engine_cost
# Breadth of coverage is bought with boundary traffic, never for free.
assert len(narrow_alerts) < len(wide_alerts) <= len(engine_alerts)


### 4. Change management

Rule content and probe content are delivered through different paths, and the
operational difference shows up at the worst possible moment — mid-incident.

| Change | Rule engine | eBPF probe |
|---|---|---|
| Add a detection | Append a rule; the engine sees the whole stream on the next event | Widen the kernel predicate and re-attach the program |
| Scope of a mistake | One rule can be disabled without touching ingestion | A bad predicate can silence the stream entirely |
| Rollback | Re-apply the previous ruleset | Re-attach the previous program build |
| Diagnostic surface | Rule text is directly comparable to the emitted event | Requires reasoning about the predicate *and* the rule |

The asymmetry to plan for: a ruleset mistake is loud and local, whereas a
predicate mistake can be silent and total. Treat predicate changes with the same
care as changing what a kernel module does.

In [ ]:
# Change-management model: apply a new detection definition and report
# what each shape can do about a bad change.

def apply_engine_ruleset(events, new_rules, old_rules):
    """Engine: ruleset swap is atomic and reversible; a bad rule is disabled
    without affecting ingestion."""
    trial = evaluate(events, new_rules)
    return {
        "alerts": trial,
        "ingestion_affected": False,
        "rollback": "re-apply previous ruleset",
        "bad_change_blast_radius": "one rule",
    }


def apply_probe_change(events, new_predicate, old_predicate):
    """Probe: predicate swap changes what exists downstream at all."""
    trial_forwarded, _ = probe_forward(events, new_predicate)
    prev_forwarded, _ = probe_forward(events, old_predicate)
    return {
        "forwarded": len(trial_forwarded),
        "ingestion_affected": len(trial_forwarded) != len(prev_forwarded),
        "rollback": "re-attach previous program build",
        "bad_change_blast_radius": "entire downstream stream",
    }


engine_result = apply_engine_ruleset(
    EVENT_STREAM,
    RULES,
    RULES,
)
probe_result = apply_probe_change(EVENT_STREAM, PROBE_PREDICATE, NARROW_PREDICATE)

print("engine swap:", engine_result["alerts"].__len__(), "alerts; ingestion affected:",
      engine_result["ingestion_affected"])
print("probe swap: ", probe_result["forwarded"], "forwarded; ingestion affected:",
      probe_result["ingestion_affected"])

assert engine_result["ingestion_affected"] is False
# The probe change is observable as a change in what downstream can even see.
assert probe_result["ingestion_affected"] is True


### 5. Scoring a concrete workload

Rather than arguing the choice in the abstract, score it. The function below
takes the properties a workload actually has and returns the recommended shape
plus the reasoning that produced it. It is deliberately simple — a weighted
score is only useful if the weights are visible and arguable.

In [ ]:
def choose_shape(workload):
    """Return (recommendation, reasons) for a workload description.

    Recognised workload keys:
      sensitive_context_needed  detection depends on fields a kernel-side
                                predicate cannot fully reproduce (full command
                                line, user identity, decoded arguments)
      volume                    rough events/second before filtering
      frequent_rule_changes     detections are expected to change often
      kernel_stability_required must run on kernels where a probe may not attach
    """
    reasons = []
    engine = 0
    probe = 0

    if workload.get("sensitive_context_needed"):
        engine += 3
        reasons.append(
            "detection needs context the engine already receives; a kernel-side "
            "predicate can only approximate it"
        )
    if workload.get("frequent_rule_changes"):
        engine += 2
        reasons.append("detections change often, and a ruleset swap is reversible "
                       "where re-attaching a program is not")
    if workload.get("kernel_stability_required"):
        engine += 2
        reasons.append("probe attachment is not guaranteed across every target kernel")

    volume = workload.get("volume", 0)
    if volume >= 1000:
        probe += 3
        reasons.append(f"~{volume} events/s makes boundary cost the dominant term")
    elif volume >= 100:
        probe += 1
        reasons.append(f"~{volume} events/s gives pre-filtering a modest benefit")
    else:
        engine += 1
        reasons.append(f"~{volume} events/s is cheap enough to carry every event "
                       "across the boundary")

    if engine == probe:
        reasons.append("scores tie; prefer the engine until a predicate proves it "
                       "is needed")
        return "rule-engine", reasons
    if engine > probe:
        reasons.append("engine score wins on context fidelity and change safety")
        return "rule-engine", reasons
    reasons.append("probe score wins on boundary cost and selectivity")
    return "ebpf-probe", reasons


CASES = [
    ("perimeter node, stable rules, low volume",
     {"volume": 20, "sensitive_context_needed": True}),
    ("noisy build node, stable broad detection",
     {"volume": 5000, "sensitive_context_needed": False}),
    ("regulated workload on mixed kernels",
     {"volume": 200, "kernel_stability_required": True}),
    ("detections iterated daily in staging",
     {"volume": 50, "sensitive_context_needed": True, "frequent_rule_changes": True}),
]

for label, workload in CASES:
    pick, why = choose_shape(workload)
    print(f"{label} -> {pick}")
    for line in why:
        print(f"  - {line}")

assert choose_shape({"volume": 5000, "sensitive_context_needed": False})[0] == "ebpf-probe"
assert choose_shape({"volume": 20, "sensitive_context_needed": True})[0] == "rule-engine"
assert choose_shape({"volume": 200, "kernel_stability_required": True,
                     "sensitive_context_needed": False})[0] == "rule-engine"
assert choose_shape({"volume": 50, "frequent_rule_changes": True,
                     "sensitive_context_needed": True})[0] == "rule-engine"


## Verify

Every code cell asserts its own behaviour, so running the notebook top to bottom
is the verification step:

- the engine cell confirms all three illustrative rules fire on the expected
  subset of the sample stream;
- the probe cell confirms the forwarded/dropped split is exhaustive and that a
  host-level detection becomes impossible once its events are filtered out;
- the cost cell confirms boundary traffic increases monotonically as the
  predicate widens, and that coverage is bought with that traffic;
- the change-management cell confirms a ruleset swap leaves ingestion untouched
  while a predicate swap is observable downstream;
- the scoring cell confirms the recommendation for four contrasting workloads.

A failure means a model assumption changed, not that the notebook is wrong —
check which relationship the assertion encodes before changing it.

## Common errors

- **Treating a narrow predicate as a complete ruleset.** A probe that only
  forwards a handful of process names is not "a faster Falco"; it is a Falco
  that can no longer see the host-level activity the dropped rules covered.
- **Encoding policy in the predicate.** Severity, priority, and suppression
  logic belong to rules. A predicate that decides *whether this matters* is a
  rule engine that runs in a place nobody reviews it.
- **Assuming a predicate is cheap to change.** Reloading rules text is a
  routine operation; re-attaching a loaded kernel program is not. Budget for
  the second one.
- **Scoring on volume alone.** High event rates argue for pre-filtering, but a
  detection that needs full context cannot be pre-filtered at all, regardless
  of how noisy the host is.
- **Reading a rule engine match as proof the event was collected.** In the probe
  model, absence of an alert may mean the event never left the kernel. Absence
  of evidence is ambiguous until the predicate's scope is known.

## References

- `falco/docs/syscall-vs-tracepoint-rules.md` — which kernel event sources a
  rule can consume, independent of where the match happens.
- `falco/docs/rule-optimization-priority-filtering.md` — priority and filtering
  behaviour on the rule-engine side.
- `falco/docs/tuned-falco-rules-noise-reduction.md` — reducing alert volume once
  the engine is carrying the stream.
- `falco/configs/container-drift-detection.yaml` — example rule configuration.
- `falco/notebooks/falco-event-output-formats.ipynb` — what happens to an event
  after it has been decided on.